In [2]:
# get 50k rows of data and store it as a dataframe
# Claude helped implement safeguards against being disconnected

import os
import requests
import pandas as pd
from dotenv import load_dotenv
import time

load_dotenv()

API_KEY = os.getenv("API_KEY")
viewName = 'cats'
limit = 250
page = 1
all_cats = []

headers = {
    "Content-Type": "application/vnd.api+json",
    "Authorization": API_KEY
}

while True:
    print(f"Pulling page {page}...")

    url = (
        f"https://api.rescuegroups.org/v5/"
        f"public/animals/"
        f"search/{viewName}/"
        f"?limit={limit}"
        f"&page={page}"
    )

    try:
        response = requests.get(url, headers=headers, timeout=30)
        response.raise_for_status()
        data = response.json()
    except Exception as e:
        print(f"Error on page {page}: {e} — retrying in 5s...")
        time.sleep(5)
        continue  # retry the same page

    for item in data['data']:
        row = {'id': item['id']}
        row.update(item['attributes'])
        all_cats.append(row)

    print(f"  Got {len(data['data'])} records. Total so far: {len(all_cats)}")

    if len(all_cats) >= 50_000:
        print("Reached 50k rows, stopping.")
        break

    if len(data['data']) < data['meta']['limit']:
        break

    page += 1
    time.sleep(1)  # be polite to the server between pages

df_cats = pd.DataFrame(all_cats)

Pulling page 1...
  Got 250 records. Total so far: 250
Pulling page 2...
  Got 250 records. Total so far: 500
Pulling page 3...
  Got 250 records. Total so far: 750
Pulling page 4...
  Got 250 records. Total so far: 1000
Pulling page 5...
  Got 250 records. Total so far: 1250
Pulling page 6...
  Got 250 records. Total so far: 1500
Pulling page 7...
  Got 250 records. Total so far: 1750
Pulling page 8...
  Got 250 records. Total so far: 2000
Pulling page 9...
  Got 250 records. Total so far: 2250
Pulling page 10...
  Got 250 records. Total so far: 2500
Pulling page 11...
  Got 250 records. Total so far: 2750
Pulling page 12...
  Got 250 records. Total so far: 3000
Pulling page 13...
  Got 250 records. Total so far: 3250
Pulling page 14...
  Got 250 records. Total so far: 3500
Pulling page 15...
  Got 250 records. Total so far: 3750
Pulling page 16...
  Got 250 records. Total so far: 4000
Pulling page 17...
  Got 250 records. Total so far: 4250
Pulling page 18...
  Got 250 records. Total

In [3]:
# change two columns to be date time type of data, and create a new column representing our dependent variable, length of stay.
df_cats['adoptedDate'] = pd.to_datetime(df_cats['adoptedDate'], errors='coerce')
df_cats['availableDate'] = pd.to_datetime(df_cats['availableDate'], errors='coerce')
df_cats['length_of_stay'] = (df_cats['adoptedDate'] - df_cats['availableDate']).dt.days
df_cats.info()

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 77 columns):
 #   Column                 Non-Null Count  Dtype              
---  ------                 --------------  -----              
 0   id                     50000 non-null  str                
 1   adoptedDate            21774 non-null  datetime64[us, UTC]
 2   isAdoptionPending      50000 non-null  bool               
 3   ageGroup               46701 non-null  str                
 4   ageString              20816 non-null  str                
 5   birthDate              20816 non-null  str                
 6   isBirthDateExact       50000 non-null  bool               
 7   breedString            50000 non-null  str                
 8   breedPrimary           50000 non-null  str                
 9   breedPrimaryId         50000 non-null  int64              
 10  breedSecondary         11862 non-null  str                
 11  breedSecondaryId       11862 non-null  float64            
 12  c

In [4]:

# df_unfiltered = pd.read_csv('../data/raw/cats.csv')
df_cats_filtered = df_cats[['id','sex','sizeCurrent','sizeGroup', 'ageGroup', 'ageString','breedString','colorDetails', 'vocalLevel', 'sheddingLevel', 'energyLevel', 'exerciseNeeds', 'isSpecialNeeds', 'isCurrentVaccinations', 'isDeclawed', 'isHousetrained', 'isKidsOk', 'adultSexesOk','obedienceTraining', 'ownerExperience', 'newPeopleReaction', 'pictureCount', 'videoCount', 'adoptedDate', 'availableDate', 'length_of_stay']]
df_cats_filtered.to_csv('../data/raw/cats_data_filtered.csv', index=False)

In [ ]:
# previously used, but too large to upload to github at the moment:
# df_cats.to_csv('../data/raw/cats.csv', index=False)
# Will use later on if necessary.

# I would also next time add sizeUOM as a column to pull to compare currentSize between animals while making sure the UOM is the same for all.